# Support Vector Machines — Practice Set 1 🎯
## eBay iPhone Auctions — Binary Classification

### Rules (same as real quiz)
- `alpha = 0.05`, `random_state = 617`
- Use `train` sample unless stated otherwise
- Do NOT round; no commas; lead with 0
- **Scale features before SVM** (always!)

### About the Data
`eBayClean.csv` — eBay auction listings for iPhones.
- `sold`: **Target** — 1 if sold, 0 if not (binary classification)
- `startprice`: Starting price
- `charCountDescription`, `upperCaseDescription`: Text features
- `biddable`, `condition`, `cellular`, `carrier`, `color`, `storage`, `productline`, `noDescription`, `startprice_99end`: Other features

> **Goal**: Build SVM classifiers to predict whether an eBay iPhone listing will sell.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV

data = pd.read_csv('eBayClean.csv')
print('Shape:', data.shape)
print('\nSold distribution:')
print(data.sold.value_counts(normalize=True).round(3))

---
## ❓ Question 1
> Separate outcome from features. Assign `sold` to `y`. Assign `startprice`, `charCountDescription`, `upperCaseDescription` to `X` (numeric features only).
>
> Split into 70% train / 30% test using `train_test_split()` with `random_state=617`.
>
> **What percentage of listings in the train sample were sold?** (Do not include % sign)

In [ ]:
y = data.sold
X = data[['startprice','charCountDescription','upperCaseDescription']]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=617)

print('Train shape:', X_train_raw.shape)
pct_sold = y_train.mean() * 100
print('% sold in train:', pct_sold)

---
## ❓ Question 2
> **Why must we scale features before fitting an SVM?**
>
> Scale the features using `StandardScaler`. Fit on train, transform both train and test.
>
> **What is the mean of `startprice` in the SCALED train set?** (Should be approximately 0)

In [ ]:
# SVMs are sensitive to feature scale — always standardize first!
scaler = StandardScaler()
scaler.fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print('Mean of scaled startprice in train:', X_train[:, 0].mean())
print('Std of scaled startprice in train: ', X_train[:, 0].std())

---
## ❓ Question 3
> Fit a **Linear SVM** classifier using `LinearSVC(C=1, random_state=617)`.
>
> **What is the train accuracy?**
> **What is the test accuracy?**

In [ ]:
svm_linear = LinearSVC(C=1, random_state=617)
svm_linear.fit(X_train, y_train)

pred_train = svm_linear.predict(X_train)
pred_test  = svm_linear.predict(X_test)

print('Train accuracy:', accuracy_score(y_train, pred_train))
print('Test accuracy: ', accuracy_score(y_test,  pred_test))

---
## ❓ Question 4
> Fit a **Non-Linear SVM** using `SVC(kernel='rbf', C=1, random_state=617)` (RBF kernel).
>
> **What is the train accuracy?**
> **What is the test accuracy?**
> **Does RBF outperform the linear SVM on test data?**

In [ ]:
svm_rbf = SVC(kernel='rbf', C=1, random_state=617)
svm_rbf.fit(X_train, y_train)

print('RBF Train accuracy:', svm_rbf.score(X_train, y_train))
print('RBF Test accuracy: ', svm_rbf.score(X_test,  y_test))

---
## ❓ Question 5
> For the **RBF SVM**, examine the effect of `gamma` on performance.
> Try `gamma` values: `[1e-3, 1e-2, 1e-1, 1, 10]`.
>
> **Which `gamma` value gives the best test accuracy?**
> **What happens to train accuracy as gamma increases?**

In [ ]:
gamma_values = [1e-3, 1e-2, 1e-1, 1, 10]
rows = []
for g in gamma_values:
    svc = SVC(kernel='rbf', gamma=g, random_state=617)
    svc.fit(X_train, y_train)
    rows.append({
        'gamma': g,
        'train_acc': svc.score(X_train, y_train),
        'test_acc' : svc.score(X_test,  y_test)
    })
df_gamma = pd.DataFrame(rows)
print(df_gamma)
print('\nBest test acc at gamma =', df_gamma.loc[df_gamma.test_acc.idxmax(), 'gamma'])

---
## ❓ Question 6
> Use `GridSearchCV` with 5-fold CV to tune BOTH `C` and `gamma` for an RBF SVM.
>
> Use: `C = [0.1, 1, 10, 100]` and `gamma = [1e-3, 1e-2, 1e-1, 1]`.
>
> **What is the best `C` and `gamma` combination?**
> **What is the best CV score?**

In [ ]:
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': [1e-3, 1e-2, 1e-1, 1]}
svc_tune = SVC(kernel='rbf', random_state=617)
grid_search = GridSearchCV(svc_tune, param_grid, cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print('Best CV score:', grid_search.best_score_)

---
## ❓ Question 7
> Using the **best model** from GridSearchCV, what are the **train and test accuracies**?

In [ ]:
print('Tuned model train accuracy:', grid_search.score(X_train, y_train))
print('Tuned model test accuracy: ', grid_search.score(X_test,  y_test))

---
## ❓ Question 8
> Print a **confusion matrix** and **classification report** for the tuned model on the test set.
>
> **What is the sensitivity (recall for class 1)?** A sold item not identified is a missed opportunity.
>
> **What is the precision for class 1?**

In [ ]:
best_pred_test = grid_search.predict(X_test)
print('Confusion Matrix:')
print(confusion_matrix(y_test, best_pred_test))
print('\nClassification Report:')
print(classification_report(y_test, best_pred_test))

---
## ❓ Question 9
> Compare the following models by **test accuracy**:
> - LinearSVC (C=1)
> - SVC RBF default (C=1, gamma='scale')
> - SVC RBF best from GridSearchCV
>
> **Rank them from best to worst.**

In [ ]:
comparison = pd.DataFrame({
    'model': ['LinearSVC C=1', 'RBF default', 'RBF tuned'],
    'test_accuracy': [
        accuracy_score(y_test, svm_linear.predict(X_test)),
        svm_rbf.score(X_test, y_test),
        grid_search.score(X_test, y_test)
    ]
}).set_index('model').sort_values('test_accuracy', ascending=False)
print(comparison)

---
## ❓ Question 10
> **Conceptual question (select all that apply):**
>
> Based on what you've observed, which of the following statements are TRUE?
>
> A. Higher gamma always improves test accuracy.
> B. SVM requires feature scaling because it is sensitive to variable scales.
> C. LinearSVC cannot be used when the number of features exceeds the number of observations.
> D. The tuned RBF SVM is expected to outperform the default RBF SVM.
> E. SVC with RBF kernel uses the kernel trick to handle non-linear boundaries.
> F. LinearSVC supports the `predict_proba()` method for class probabilities.

In [ ]:
# Answers:
# A. FALSE — high gamma causes overfitting (high train, low test acc)
# B. TRUE  — SVM uses distances, so scale matters
# C. FALSE — LinearSVC can be used; it actually works well when n < p
# D. TRUE  — GridSearchCV finds optimal C and gamma
# E. TRUE  — RBF kernel projects to higher-dimensional space implicitly
# F. FALSE — SVM does NOT have predict_proba() by default

# CORRECT: B, D, E
print('Correct answers: B, D, E')

---
# ✅ Key SVM Concepts

| Concept | Detail |
|---|---|
| **Must scale** | `StandardScaler().fit(X_train)` then transform both |
| **LinearSVC** | Linear kernel, fast, no `predict_proba()` |
| **SVC(kernel='rbf')** | Non-linear via kernel trick |
| **C parameter** | Higher C = narrower margin = more overfitting risk |
| **gamma** | Higher gamma = more complex boundary = overfitting risk |
| **GridSearchCV** | Tune C and gamma via cross-validation |
| **No probabilities** | SVM doesn't natively output `predict_proba()` |

## SVM Pipeline (必记！)
```python
# 1. Split
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X, y, train_size=0.7, random_state=617)

# 2. Scale (fit on train only!)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_te = scaler.transform(X_te_raw)   # only transform, never fit on test

# 3. Fit
svm = SVC(kernel='rbf', C=1, random_state=617)
svm.fit(X_tr, y_tr)

# 4. Evaluate
print(svm.score(X_tr, y_tr))   # train
print(svm.score(X_te, y_te))   # test
```